In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

## Functions

In [3]:
deconv_map = {"xgb": "XGB", "mlp": "MLP", "swn": "SWN", "nnls": "NNLS", "psls": "PSLS"}
calib_map = {
    "uncalibrated": "None",
    "linear_clip_normalize": "Lin.\\ clip+norm",
    "linear_simplex_projection": "Lin.\\ simplex",
    "vector_scaling": "Vec.\\ scaling",
}

deconv_order = ["xgb", "mlp", "swn", "nnls", "psls"]
calib_order = list(calib_map.keys())
n_calib = len(calib_order)
n_total = len(deconv_order) * n_calib


def format_deconvolver_results(name, metrics_):
    lines = []
    first_row = True
    for i, dec in enumerate(deconv_order):
        for j, cal in enumerate(calib_order):
            row = metrics_[
                (metrics_["deconvolver"] == dec)
                & (metrics_["calibration_method"] == cal)
            ].iloc[0]

            r2 = f"{row['r2'] * 100:.2f}"
            r2_ci = f" {{\\scriptsize[{row['r2_ci_lower']*100:.2f}, {row['r2_ci_upper']*100:.2f}]}}"
            loa = f"[{row['loa_lower']*1e2:.2f}, {row['loa_upper']*1e2:.2f}]"
            loa_worst = f"[{row['worst_class_loa_lower']*1e2:.2f}, {row['worst_class_loa_upper']*1e2:.2f}]"
            mae = f"{row['mae']*1e3:.2f}"
            mae_ci = f" {{\\scriptsize[{row['mae_ci_lower']*1e3:.2f}, {row['mae_ci_upper']*1e3:.2f}]}}"
            mse = f"{row['mse']*1e4:.2f}"
            mse_ci = f" {{\\scriptsize[{row['mse_ci_lower']*1e4:.2f}, {row['mse_ci_upper']*1e4:.2f}]}}"
            kl = f"{row['kl']*1e2:.2f}"
            kl_ci = f" {{\\scriptsize[{row['kl_ci_lower']*1e2:.2f}, {row['kl_ci_upper']*1e2:.2f}]}}"

            cal_label = calib_map[cal]

            if first_row:
                lines.append(
                    f"                             \\multirow{{{n_total}}}{{*}}{{{name}}}"
                )
                lines.append(
                    f"                             & \\multirow{{{n_calib}}}{{*}}{{{deconv_map[dec]}}}"
                    f"              & {cal_label:<25s} & {r2 + r2_ci:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae + mae_ci:<12s} & {mse + mse_ci:<12s} & {kl + kl_ci} \\\\"
                )
                first_row = False
            elif j == 0:
                lines.append(
                    f"                             "
                    f"& \\multirow{{{n_calib}}}{{*}}{{{deconv_map[dec]}}}              & {cal_label:<25s} & {r2 + r2_ci:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae + mae_ci:<12s} & {mse + mse_ci:<12s} & {kl + kl_ci} \\\\"
                )
            else:
                lines.append(
                    f"                             "
                    f"&                                   & {cal_label:<25s} & {r2 + r2_ci:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae + mae_ci:<12s} & {mse + mse_ci:<12s} & {kl + kl_ci} \\\\"
                )

        if i < len(deconv_order) - 1:
            lines.append("        \\cmidrule(l){2-9}")

    return "\n".join(lines)

## Soft labels with pooling

In [8]:
metrics_dismir_softlabels_with_pooling = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/1f0aaa3947bb4732a945e76aed3a5864/artifacts/calibration/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("Dismir", metrics_dismir_softlabels_with_pooling))
del metrics_dismir_softlabels_with_pooling

                             \multirow{20}{*}{Dismir}
                             & \multirow{4}{*}{XGB}              & None                      & 97.45 {\scriptsize[97.40, 97.49]} & [-2.85, 2.85]       & [-7.32, 8.31]           & 3.67 {\scriptsize[3.65, 3.69]} & 2.12 {\scriptsize[2.08, 2.16]} & 6.35 {\scriptsize[6.31, 6.39]} \\
                             &                                   & Lin.\ clip+norm           & 97.74 {\scriptsize[97.70, 97.78]} & [-2.68, 2.68]       & [-7.16, 7.97]           & 3.37 {\scriptsize[3.36, 3.39]} & 1.88 {\scriptsize[1.84, 1.91]} & 6.16 {\scriptsize[6.11, 6.21]} \\
                             &                                   & Lin.\ simplex             & 98.07 {\scriptsize[98.03, 98.10]} & [-2.48, 2.48]       & [-6.96, 7.69]           & 3.07 {\scriptsize[3.05, 3.08]} & 1.60 {\scriptsize[1.57, 1.64]} & 6.43 {\scriptsize[6.37, 6.49]} \\
                             &                                   & Vec.\ scaling             & 97.74 {\script

In [9]:
metrics_methylbert_softlabels_with_pooling = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/e21d054a7dc04dd2a7e8de4231efaa56/artifacts/calibration/top_156_features/confidence_intervals_summary.csv"
)
print(
    format_deconvolver_results("MethylBERT", metrics_methylbert_softlabels_with_pooling)
)
del metrics_methylbert_softlabels_with_pooling

                             \multirow{20}{*}{MethylBERT}
                             & \multirow{4}{*}{XGB}              & None                      & 97.27 {\scriptsize[97.23, 97.31]} & [-2.95, 2.95]       & [-7.33, 7.51]           & 3.82 {\scriptsize[3.80, 3.84]} & 2.27 {\scriptsize[2.23, 2.30]} & 6.51 {\scriptsize[6.47, 6.55]} \\
                             &                                   & Lin.\ clip+norm           & 97.70 {\scriptsize[97.66, 97.73]} & [-2.71, 2.71]       & [-6.78, 7.27]           & 3.43 {\scriptsize[3.42, 3.45]} & 1.91 {\scriptsize[1.88, 1.94]} & 6.08 {\scriptsize[6.03, 6.12]} \\
                             &                                   & Lin.\ simplex             & 98.09 {\scriptsize[98.06, 98.13]} & [-2.46, 2.46]       & [-6.46, 6.88]           & 3.07 {\scriptsize[3.06, 3.09]} & 1.58 {\scriptsize[1.55, 1.61]} & 6.22 {\scriptsize[6.16, 6.27]} \\
                             &                                   & Vec.\ scaling             & 97.65 {\sc

In [10]:
metrics_lookup_softlabels_with_pooling = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/6600218996b74b3cb938d09f05ca362b/artifacts/calibration//confidence_intervals_summary.csv"
)
print(
    format_deconvolver_results(
        "Lookup Classifier", metrics_lookup_softlabels_with_pooling
    )
)
del metrics_lookup_softlabels_with_pooling

                             \multirow{20}{*}{Lookup Classifier}
                             & \multirow{4}{*}{XGB}              & None                      & 90.65 {\scriptsize[90.54, 90.77]} & [-5.46, 5.46]       & [-17.10, 15.16]         & 7.26 {\scriptsize[7.23, 7.30]} & 7.76 {\scriptsize[7.66, 7.86]} & 19.58 {\scriptsize[19.42, 19.74]} \\
                             &                                   & Lin.\ clip+norm           & 92.47 {\scriptsize[92.37, 92.56]} & [-4.90, 4.90]       & [-16.58, 15.16]         & 6.27 {\scriptsize[6.24, 6.30]} & 6.25 {\scriptsize[6.17, 6.34]} & 18.08 {\scriptsize[17.91, 18.25]} \\
                             &                                   & Lin.\ simplex             & 93.31 {\scriptsize[93.22, 93.41]} & [-4.62, 4.62]       & [-17.09, 14.29]         & 5.61 {\scriptsize[5.58, 5.64]} & 5.55 {\scriptsize[5.47, 5.63]} & 22.98 {\scriptsize[22.70, 23.28]} \\
                             &                                   & Vec.\ scaling         

## Soft labels without pooling

In [7]:
metrics_dismir_softlabels_without_pooling = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/1ca99b1b1b8e44beb62978c0701ca6fd/artifacts/calibration/top_156_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("Dismir", metrics_dismir_softlabels_without_pooling))
del metrics_dismir_softlabels_without_pooling

                             \multirow{20}{*}{Dismir}
                             & \multirow{4}{*}{XGB}              & None                      & 97.08 {\scriptsize[97.03, 97.13]} & [-3.05, 3.05]       & [-10.44, 10.21]         & 3.77 {\scriptsize[3.75, 3.79]} & 2.42 {\scriptsize[2.38, 2.46]} & 6.72 {\scriptsize[6.67, 6.77]} \\
                             &                                   & Lin.\ clip+norm           & 97.40 {\scriptsize[97.35, 97.44]} & [-2.88, 2.88]       & [-9.94, 9.94]           & 3.45 {\scriptsize[3.43, 3.47]} & 2.16 {\scriptsize[2.12, 2.20]} & 6.41 {\scriptsize[6.35, 6.46]} \\
                             &                                   & Lin.\ simplex             & 97.71 {\scriptsize[97.67, 97.75]} & [-2.70, 2.70]       & [-9.77, 9.59]           & 3.15 {\scriptsize[3.14, 3.17]} & 1.90 {\scriptsize[1.87, 1.94]} & 6.68 {\scriptsize[6.62, 6.75]} \\
                             &                                   & Vec.\ scaling             & 97.51 {\script

In [4]:
metrics_methylbert_softlabels_without_pooling = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/c7f23c68ba774768a99f1861d0ac8581/artifacts/calibration/top_156_features/confidence_intervals_summary.csv"
)
print(
    format_deconvolver_results(
        "MethylBERT", metrics_methylbert_softlabels_without_pooling
    )
)
del metrics_methylbert_softlabels_without_pooling

                             \multirow{20}{*}{MethylBERT}
                             & \multirow{4}{*}{XGB}              & None                      & 95.98 {\scriptsize[95.92, 96.03]} & [-3.58, 3.58]       & [-10.20, 8.45]          & 4.68 {\scriptsize[4.66, 4.70]} & 3.34 {\scriptsize[3.29, 3.39]} & 9.20 {\scriptsize[9.14, 9.26]} \\
                             &                                   & Lin.\ clip+norm           & 96.85 {\scriptsize[96.81, 96.90]} & [-3.17, 3.17]       & [-6.57, 7.05]           & 4.10 {\scriptsize[4.08, 4.12]} & 2.61 {\scriptsize[2.57, 2.65]} & 8.30 {\scriptsize[8.24, 8.36]} \\
                             &                                   & Lin.\ simplex             & 97.45 {\scriptsize[97.41, 97.49]} & [-2.85, 2.85]       & [-5.96, 6.60]           & 3.64 {\scriptsize[3.62, 3.65]} & 2.12 {\scriptsize[2.08, 2.15]} & 9.56 {\scriptsize[9.46, 9.66]} \\
                             &                                   & Vec.\ scaling             & 96.90 {\sc

In [8]:
metrics_lookup_softlabels_without_pooling = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/6bbcdb4f26eb40b1b6cd8829c272883b/artifacts/calibration//confidence_intervals_summary.csv"
)
print(
    format_deconvolver_results(
        "Lookup Classifier", metrics_lookup_softlabels_without_pooling
    )
)
del metrics_lookup_softlabels_without_pooling

                             \multirow{20}{*}{Lookup Classifier}
                             & \multirow{4}{*}{XGB}              & None                      & 76.78 {\scriptsize[76.60, 76.96]} & [-8.60, 8.60]       & [-19.02, 16.31]         & 14.89 {\scriptsize[14.85, 14.93]} & 19.27 {\scriptsize[19.09, 19.46]} & 45.76 {\scriptsize[45.55, 45.98]} \\
                             &                                   & Lin.\ clip+norm           & 85.35 {\scriptsize[85.17, 85.52]} & [-6.84, 6.84]       & [-18.20, 17.52]         & 10.25 {\scriptsize[10.22, 10.29]} & 12.16 {\scriptsize[12.00, 12.33]} & 31.67 {\scriptsize[31.45, 31.90]} \\
                             &                                   & Lin.\ simplex             & 87.97 {\scriptsize[87.79, 88.15]} & [-6.19, 6.19]       & [-19.03, 16.54]         & 8.31 {\scriptsize[8.27, 8.35]} & 9.99 {\scriptsize[9.83, 10.15]} & 33.51 {\scriptsize[33.17, 33.85]} \\
                             &                                   & Vec.\ sca

## Hard labels diagrej

In [6]:
metrics_dismir_hardlabels_diagrej = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/5b0cc3ebb14d40788b11978a691f41f8/artifacts/calibration/diag_background_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("Dismir", metrics_dismir_hardlabels_diagrej))
del metrics_dismir_hardlabels_diagrej

                             \multirow{20}{*}{Dismir}
                             & \multirow{4}{*}{XGB}              & None                      & 96.82 {\scriptsize[96.76, 96.88]} & [-3.18, 3.18]       & [-7.75, 8.34]           & 3.99 {\scriptsize[3.97, 4.01]} & 2.64 {\scriptsize[2.59, 2.69]} & 7.13 {\scriptsize[7.08, 7.18]} \\
                             &                                   & Lin.\ clip+norm           & 97.24 {\scriptsize[97.18, 97.29]} & [-2.97, 2.97]       & [-7.38, 8.22]           & 3.59 {\scriptsize[3.58, 3.61]} & 2.29 {\scriptsize[2.25, 2.34]} & 6.77 {\scriptsize[6.71, 6.82]} \\
                             &                                   & Lin.\ simplex             & 97.66 {\scriptsize[97.61, 97.71]} & [-2.73, 2.73]       & [-7.19, 7.95]           & 3.20 {\scriptsize[3.18, 3.22]} & 1.94 {\scriptsize[1.90, 1.99]} & 6.99 {\scriptsize[6.92, 7.06]} \\
                             &                                   & Vec.\ scaling             & 97.51 {\script

In [5]:
metrics_methylbert_hardlabels_diagrej = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/a3f1f111ffc14085bc70c8bf60a78e4d/artifacts/calibration/diag_background_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("MethylBERT", metrics_methylbert_hardlabels_diagrej))
del metrics_methylbert_hardlabels_diagrej

                             \multirow{20}{*}{MethylBERT}
                             & \multirow{4}{*}{XGB}              & None                      & 95.26 {\scriptsize[95.19, 95.34]} & [-3.89, 3.89]       & [-8.46, 9.33]           & 4.88 {\scriptsize[4.85, 4.90]} & 3.93 {\scriptsize[3.86, 4.00]} & 10.09 {\scriptsize[10.01, 10.16]} \\
                             &                                   & Lin.\ clip+norm           & 96.05 {\scriptsize[95.98, 96.11]} & [-3.55, 3.55]       & [-8.45, 8.82]           & 4.24 {\scriptsize[4.22, 4.27]} & 3.28 {\scriptsize[3.22, 3.34]} & 9.87 {\scriptsize[9.79, 9.95]} \\
                             &                                   & Lin.\ simplex             & 96.70 {\scriptsize[96.64, 96.76]} & [-3.24, 3.24]       & [-8.33, 8.54]           & 3.78 {\scriptsize[3.76, 3.80]} & 2.74 {\scriptsize[2.68, 2.79]} & 11.48 {\scriptsize[11.38, 11.58]} \\
                             &                                   & Vec.\ scaling             & 97.1

In [4]:
metrics_lookup_hardlabels_diagrej = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/1e4280271d584868a0c816512ea995a4/artifacts/calibration/diag_background_features/confidence_intervals_summary.csv"
)
print(
    format_deconvolver_results("Lookup Classifier", metrics_lookup_hardlabels_diagrej)
)
del metrics_lookup_hardlabels_diagrej

                             \multirow{20}{*}{Lookup Classifier}
                             & \multirow{4}{*}{XGB}              & None                      & 82.72 {\scriptsize[82.60, 82.84]} & [-7.42, 7.42]       & [-16.41, 13.17]         & 12.13 {\scriptsize[12.10, 12.17]} & 14.34 {\scriptsize[14.21, 14.48]} & 38.60 {\scriptsize[38.37, 38.83]} \\
                             &                                   & Lin.\ clip+norm           & 86.76 {\scriptsize[86.66, 86.86]} & [-6.50, 6.50]       & [-15.20, 12.19]         & 10.04 {\scriptsize[10.00, 10.07]} & 10.99 {\scriptsize[10.88, 11.09]} & 44.44 {\scriptsize[44.07, 44.81]} \\
                             &                                   & Lin.\ simplex             & 88.46 {\scriptsize[88.36, 88.56]} & [-6.07, 6.07]       & [-13.52, 13.91]         & 8.86 {\scriptsize[8.82, 8.89]} & 9.58 {\scriptsize[9.49, 9.67]} & 54.70 {\scriptsize[54.19, 55.21]} \\
                             &                                   & Vec.\ scal

## Hard labels using top156 features

In [11]:
metrics_dismir_hardlabels_top156 = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/5b0cc3ebb14d40788b11978a691f41f8/artifacts/calibration/top_156_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("Dismir", metrics_dismir_hardlabels_top156))
del metrics_dismir_hardlabels_top156

                             \multirow{20}{*}{Dismir}
                             & \multirow{4}{*}{XGB}              & None                      & 96.99 {\scriptsize[96.93, 97.05]} & [-3.10, 3.10]       & [-8.21, 8.72]           & 3.79 {\scriptsize[3.77, 3.81]} & 2.50 {\scriptsize[2.45, 2.55]} & 6.69 {\scriptsize[6.64, 6.74]} \\
                             &                                   & Lin.\ clip+norm           & 97.36 {\scriptsize[97.31, 97.41]} & [-2.90, 2.90]       & [-7.86, 8.57]           & 3.45 {\scriptsize[3.43, 3.47]} & 2.19 {\scriptsize[2.14, 2.24]} & 6.47 {\scriptsize[6.42, 6.53]} \\
                             &                                   & Lin.\ simplex             & 97.73 {\scriptsize[97.68, 97.78]} & [-2.69, 2.69]       & [-7.67, 8.26]           & 3.10 {\scriptsize[3.08, 3.11]} & 1.88 {\scriptsize[1.84, 1.93]} & 6.72 {\scriptsize[6.65, 6.79]} \\
                             &                                   & Vec.\ scaling             & 97.60 {\script

In [12]:
metrics_methylbert_hardlabels_top156 = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/a3f1f111ffc14085bc70c8bf60a78e4d/artifacts/calibration/top_156_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("MethylBERT", metrics_methylbert_hardlabels_top156))
del metrics_methylbert_hardlabels_top156

                             \multirow{20}{*}{MethylBERT}
                             & \multirow{4}{*}{XGB}              & None                      & 95.10 {\scriptsize[95.03, 95.17]} & [-3.95, 3.95]       & [-9.21, 9.80]           & 4.98 {\scriptsize[4.96, 5.01]} & 4.07 {\scriptsize[4.00, 4.13]} & 10.16 {\scriptsize[10.09, 10.22]} \\
                             &                                   & Lin.\ clip+norm           & 95.95 {\scriptsize[95.88, 96.01]} & [-3.60, 3.60]       & [-9.22, 9.51]           & 4.38 {\scriptsize[4.36, 4.40]} & 3.37 {\scriptsize[3.31, 3.42]} & 9.87 {\scriptsize[9.80, 9.94]} \\
                             &                                   & Lin.\ simplex             & 96.51 {\scriptsize[96.45, 96.56]} & [-3.34, 3.34]       & [-9.20, 9.22]           & 3.96 {\scriptsize[3.94, 3.98]} & 2.90 {\scriptsize[2.85, 2.95]} & 11.33 {\scriptsize[11.23, 11.43]} \\
                             &                                   & Vec.\ scaling             & 96.5

In [13]:
metrics_lookup_hardlabels_top156 = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/1e4280271d584868a0c816512ea995a4/artifacts/calibration/top_156_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("Lookup Classifier", metrics_lookup_hardlabels_top156))
del metrics_lookup_hardlabels_top156

                             \multirow{20}{*}{Lookup Classifier}
                             & \multirow{4}{*}{XGB}              & None                      & 82.72 {\scriptsize[82.60, 82.84]} & [-7.42, 7.42]       & [-16.41, 13.17]         & 12.13 {\scriptsize[12.10, 12.17]} & 14.34 {\scriptsize[14.21, 14.48]} & 38.60 {\scriptsize[38.37, 38.83]} \\
                             &                                   & Lin.\ clip+norm           & 86.76 {\scriptsize[86.66, 86.86]} & [-6.50, 6.50]       & [-15.20, 12.19]         & 10.04 {\scriptsize[10.00, 10.07]} & 10.99 {\scriptsize[10.88, 11.09]} & 44.44 {\scriptsize[44.07, 44.81]} \\
                             &                                   & Lin.\ simplex             & 88.46 {\scriptsize[88.36, 88.56]} & [-6.07, 6.07]       & [-13.52, 13.91]         & 8.86 {\scriptsize[8.82, 8.89]} & 9.58 {\scriptsize[9.49, 9.67]} & 54.70 {\scriptsize[54.19, 55.21]} \\
                             &                                   & Vec.\ scal

## CancerDetector

In [6]:
metrics_cancerdetector_trainfreq = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/87dd6700cdd34d37bd01a8d2931a1dd4/artifacts/calibration/top_156_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("Training Counts", metrics_cancerdetector_trainfreq))
del metrics_cancerdetector_trainfreq

                             \multirow{20}{*}{Training Counts}
                             & \multirow{4}{*}{XGB}              & None                      & 97.32 {\scriptsize[97.27, 97.37]} & [-2.92, 2.92]       & [-8.14, 8.45]           & 3.68 {\scriptsize[3.66, 3.70]} & 2.22 {\scriptsize[2.18, 2.26]} & 6.47 {\scriptsize[6.43, 6.52]} \\
                             &                                   & Lin.\ clip+norm           & 97.66 {\scriptsize[97.62, 97.71]} & [-2.73, 2.73]       & [-8.07, 8.14]           & 3.37 {\scriptsize[3.35, 3.38]} & 1.94 {\scriptsize[1.90, 1.98]} & 6.47 {\scriptsize[6.41, 6.52]} \\
                             &                                   & Lin.\ simplex             & 97.96 {\scriptsize[97.92, 98.00]} & [-2.55, 2.55]       & [-8.00, 7.93]           & 3.08 {\scriptsize[3.07, 3.10]} & 1.69 {\scriptsize[1.65, 1.73]} & 7.17 {\scriptsize[7.10, 7.25]} \\
                             &                                   & Vec.\ scaling             & 97.60

In [7]:
metrics_cancerdetector_uniform = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/aa2a6b70f9c94befa49b1b30ed2cecd6/artifacts/calibration/top_156_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("Uniform", metrics_cancerdetector_uniform))
del metrics_cancerdetector_uniform

                             \multirow{20}{*}{Uniform}
                             & \multirow{4}{*}{XGB}              & None                      & 97.31 {\scriptsize[97.26, 97.36]} & [-2.93, 2.93]       & [-7.76, 8.08]           & 3.68 {\scriptsize[3.66, 3.70]} & 2.23 {\scriptsize[2.19, 2.27]} & 6.51 {\scriptsize[6.46, 6.55]} \\
                             &                                   & Lin.\ clip+norm           & 97.67 {\scriptsize[97.62, 97.71]} & [-2.73, 2.73]       & [-7.50, 7.79]           & 3.36 {\scriptsize[3.34, 3.37]} & 1.94 {\scriptsize[1.90, 1.98]} & 6.42 {\scriptsize[6.37, 6.47]} \\
                             &                                   & Lin.\ simplex             & 97.98 {\scriptsize[97.94, 98.03]} & [-2.54, 2.54]       & [-7.33, 7.51]           & 3.06 {\scriptsize[3.05, 3.07]} & 1.67 {\scriptsize[1.64, 1.71]} & 7.06 {\scriptsize[6.99, 7.13]} \\
                             &                                   & Vec.\ scaling             & 97.68 {\scrip

## Baselines

In [5]:
metrics_baselines = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/1f0aaa3947bb4732a945e76aed3a5864/artifacts/basleines_calibration/confidence_intervals_summary.csv"
)
del metrics_uxm

In [6]:
metrics_baselines

,deconvolver,calibration_method,cosine_sim,kl,kl_ci_lower,kl_ci_upper,kl_per_sample,loa_lower,loa_upper,loa_width,...,mse_per_sample,per_class_loa,r2,r2_ci_lower,r2_ci_upper,worst_class_idx,worst_class_loa_lower,worst_class_loa_upper,worst_class_loa_width,worst_class_name
0,celfie,uncalibrated,0.985819,0.142883,0.142346,0.143414,[0.35898172 0.20103254 0.07424278 ... 0.065567...,-0.035490,0.035490,0.070981,...,[2.95465178e-03 2.47281938e-04 4.97946349e-05 ...,"{'bias': array([-3.16715118e-03, 5.35973303e-...",0.960495,0.960035,0.960957,11,-0.099610,0.091548,0.191158,Colon-Fibro
1,celfie,linear_clip_normalize,0.988055,0.104440,0.103616,0.105281,[0.19832193 0.23103672 0.01646339 ... 0.016679...,-0.028711,0.028711,0.057422,...,[1.05101213e-03 2.10770171e-04 1.73178878e-05 ...,"{'bias': array([-3.82240946e-03, 1.46718051e-...",0.974146,0.973853,0.974440,11,-0.063044,0.079813,0.142857,Colon-Fibro
2,celfie,linear_simplex_projection,0.988732,0.113143,0.111910,0.114381,[0. 0.21256289 0.01007689 ... 0.063322...,-0.025032,0.025032,0.050064,...,[0.00000000e+00 2.63838484e-04 1.85950272e-05 ...,"{'bias': array([-0.00433181, 0.00155276, -0.0...",0.980348,0.980117,0.980576,11,-0.052514,0.071121,0.123636,Colon-Fibro
3,celfie,vector_scaling,0.979158,0.085912,0.085502,0.086329,[0.04880686 0.12623646 0.04573549 ... 0.037275...,-0.034088,0.034088,0.068177,...,[7.56603084e-05 3.45846690e-04 8.74393483e-05 ...,"{'bias': array([-4.30127913e-03, 2.58481492e-...",0.963555,0.963241,0.963873,37,-0.058519,0.085217,0.143735,Smooth-Musc
4,epidish,uncalibrated,0.962231,0.253719,0.252507,0.254914,[1.29558853 0.54821983 0.1197334 ... 0.141955...,-0.058105,0.058105,0.116210,...,[0.01447099 0.00080329 0.00012029 ... 0.000225...,"{'bias': array([-1.08162002e-02, -3.75476150e-...",0.894111,0.892837,0.895384,11,-0.157937,0.141329,0.299266,Colon-Fibro
5,epidish,linear_clip_normalize,0.979249,0.165665,0.164750,0.166559,[0.82625319 0.34413849 0.03884651 ... 0.053528...,-0.045427,0.045427,0.090855,...,[8.88007494e-03 3.12387073e-04 2.04095277e-05 ...,"{'bias': array([-4.45394992e-03, 2.03712482e-...",0.935276,0.934488,0.936093,11,-0.111680,0.109535,0.221216,Colon-Fibro
6,epidish,linear_simplex_projection,0.982870,0.127451,0.126596,0.128320,[0.49450819 0.27437795 0.02333894 ... 0.066972...,-0.036149,0.036149,0.072298,...,[4.71725446e-03 1.94044989e-04 1.56492326e-05 ...,"{'bias': array([-4.42817220e-03, 2.88889139e-...",0.959015,0.958477,0.959562,11,-0.098586,0.093691,0.192278,Colon-Fibro
7,epidish,vector_scaling,0.977070,0.105377,0.104914,0.105833,[0.10309505 0.23627102 0.06954151 ... 0.038961...,-0.035931,0.035931,0.071861,...,[0.00028032 0.00015636 0.00014724 ... 0.000131...,"{'bias': array([-6.96305371e-03, 7.99701171e-...",0.959509,0.959186,0.959849,11,-0.078544,0.067535,0.146079,Colon-Fibro
8,epidish_houseman,uncalibrated,0.964769,0.229162,0.228100,0.230196,[0.88149795 0.51263202 0.14123143 ... 0.173707...,-0.053743,0.053743,0.107485,...,[0.01025439 0.0007265 0.0001508 ... 0.000291...,"{'bias': array([-1.12317894e-02, -2.63995710e-...",0.909414,0.908312,0.910512,11,-0.155749,0.132245,0.287995,Colon-Fibro
9,epidish_houseman,linear_clip_normalize,0.979949,0.168769,0.167869,0.169646,[0.65221181 0.33705473 0.06051734 ... 0.099356...,-0.043092,0.043092,0.086183,...,[6.90959584e-03 2.90227627e-04 3.96779831e-05 ...,"{'bias': array([-6.10529849e-03, 8.88731038e-...",0.941761,0.941101,0.942448,11,-0.107135,0.102791,0.209926,Colon-Fibro


In [19]:
lines = []
for i, dec in enumerate(metrics_baselines["deconvolver"].unique()):
    for j, cal in enumerate(calib_order):
        row = metrics_baselines[
            (metrics_baselines["deconvolver"] == dec)
            & (metrics_baselines["calibration_method"] == cal)
        ].iloc[0]

        r2 = f"{row['r2'] * 100:.2f}"
        r2_ci = f" {{\\scriptsize[{row['r2_ci_lower']*100:.2f}, {row['r2_ci_upper']*100:.2f}]}}"
        loa = f"[{row['loa_lower']*1e2:.2f}, {row['loa_upper']*1e2:.2f}]"
        loa_worst = f"[{row['worst_class_loa_lower']*1e2:.2f}, {row['worst_class_loa_upper']*1e2:.2f}]"
        mae = f"{row['mae']*1e3:.2f}"
        mae_ci = f" {{\\scriptsize[{row['mae_ci_lower']*1e3:.2f}, {row['mae_ci_upper']*1e3:.2f}]}}"
        mse = f"{row['mse']*1e4:.2f}"
        mse_ci = f" {{\\scriptsize[{row['mse_ci_lower']*1e4:.2f}, {row['mse_ci_upper']*1e4:.2f}]}}"
        kl = f"{row['kl']*1e2:.2f}"
        kl_ci = f" {{\\scriptsize[{row['kl_ci_lower']*1e2:.2f}, {row['kl_ci_upper']*1e2:.2f}]}}"

        cal_label = calib_map[cal]

        if j == 0:
            lines.append(
                f"\\multirow{{{n_calib}}}{{*}}{{{dec}}}              & {cal_label:<25s} & {r2 + r2_ci:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae + mae_ci:<12s} & {mse + mse_ci:<12s} & {kl + kl_ci} \\\\"
            )
        else:
            lines.append(
                f"                                  & {cal_label:<25s} & {r2 + r2_ci:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae + mae_ci:<12s} & {mse + mse_ci:<12s} & {kl + kl_ci} \\\\"
            )

    if i < len(deconv_order) - 1:
        lines.append("\\midrule")
print("\n".join(lines))

del metrics_baselines

\multirow{4}{*}{celfie}              & None                      & 96.05 {\scriptsize[96.00, 96.10]} & [-3.55, 3.55]       & [-9.96, 9.15]           & 6.42 {\scriptsize[6.40, 6.44]} & 3.28 {\scriptsize[3.24, 3.32]} & 14.29 {\scriptsize[14.23, 14.34]} \\
                                  & Lin.\ clip+norm           & 97.41 {\scriptsize[97.39, 97.44]} & [-2.87, 2.87]       & [-6.30, 7.98]           & 4.33 {\scriptsize[4.31, 4.34]} & 2.15 {\scriptsize[2.12, 2.17]} & 10.44 {\scriptsize[10.36, 10.53]} \\
                                  & Lin.\ simplex             & 98.03 {\scriptsize[98.01, 98.06]} & [-2.50, 2.50]       & [-5.25, 7.11]           & 3.47 {\scriptsize[3.45, 3.48]} & 1.63 {\scriptsize[1.61, 1.65]} & 11.31 {\scriptsize[11.19, 11.44]} \\
                                  & Vec.\ scaling             & 96.36 {\scriptsize[96.32, 96.39]} & [-3.41, 3.41]       & [-5.85, 8.52]           & 5.52 {\scriptsize[5.50, 5.54]} & 3.02 {\scriptsize[3.01, 3.04]} & 8.59 {\scriptsize[8.55, 8.63]}

## Canonical soft labels diagrej

In [10]:
metrics_dismir_canonicalsoft_diagrej = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/627979e8835a4facb55b4185f6cdf3c2/artifacts/calibration/diag_background_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("Dismir", metrics_dismir_canonicalsoft_diagrej))
del metrics_dismir_canonicalsoft_diagrej

                             \multirow{20}{*}{Dismir}
                             & \multirow{4}{*}{XGB}              & None                      & 96.95 {\scriptsize[96.90, 96.99]} & [-3.12, 3.12]       & [-7.48, 8.33]           & 4.00 {\scriptsize[3.98, 4.02]} & 2.54 {\scriptsize[2.49, 2.58]} & 6.90 {\scriptsize[6.85, 6.94]} \\
                             &                                   & Lin.\ clip+norm           & 97.37 {\scriptsize[97.32, 97.41]} & [-2.90, 2.90]       & [-7.21, 8.18]           & 3.61 {\scriptsize[3.59, 3.63]} & 2.18 {\scriptsize[2.14, 2.22]} & 6.50 {\scriptsize[6.45, 6.55]} \\
                             &                                   & Lin.\ simplex             & 97.76 {\scriptsize[97.71, 97.80]} & [-2.67, 2.67]       & [-7.04, 7.90]           & 3.24 {\scriptsize[3.23, 3.26]} & 1.86 {\scriptsize[1.83, 1.90]} & 6.65 {\scriptsize[6.59, 6.72]} \\
                             &                                   & Vec.\ scaling             & 97.52 {\script

In [5]:
metrics_methylbert_canonicalsoft_diagrej = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/cff80360fd4e412d85a39d517de0040a/artifacts/calibration/diag_background_features/confidence_intervals_summary.csv"
)
print(
    format_deconvolver_results("MethylBERT", metrics_methylbert_canonicalsoft_diagrej)
)
del metrics_methylbert_canonicalsoft_diagrej

                             \multirow{20}{*}{MethylBERT}
                             & \multirow{4}{*}{XGB}              & None                      & 95.37 {\scriptsize[95.29, 95.44]} & [-3.84, 3.84]       & [-8.81, 9.27]           & 4.83 {\scriptsize[4.80, 4.85]} & 3.84 {\scriptsize[3.78, 3.91]} & 9.83 {\scriptsize[9.76, 9.90]} \\
                             &                                   & Lin.\ clip+norm           & 96.07 {\scriptsize[96.00, 96.13]} & [-3.54, 3.54]       & [-8.38, 8.95]           & 4.22 {\scriptsize[4.20, 4.25]} & 3.26 {\scriptsize[3.20, 3.33]} & 9.55 {\scriptsize[9.47, 9.62]} \\
                             &                                   & Lin.\ simplex             & 96.75 {\scriptsize[96.69, 96.81]} & [-3.22, 3.22]       & [-8.07, 8.52]           & 3.72 {\scriptsize[3.70, 3.74]} & 2.69 {\scriptsize[2.64, 2.75]} & 10.76 {\scriptsize[10.66, 10.86]} \\
                             &                                   & Vec.\ scaling             & 97.09 {

## Canonical soft labels top156

In [ ]:
metrics_dismir_canonicalsoft_top156 = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/627979e8835a4facb55b4185f6cdf3c2/artifacts/calibration/top_156_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("Dismir", metrics_dismir_canonicalsoft_top156))
del metrics_dismir_canonicalsoft_top156

                             \multirow{20}{*}{Dismir}
                             & \multirow{4}{*}{XGB}              & None                      & 97.15 {\scriptsize[97.10, 97.19]} & [-3.02, 3.02]       & [-7.43, 8.27]           & 3.89 {\scriptsize[3.87, 3.91]} & 2.37 {\scriptsize[2.33, 2.41]} & 6.68 {\scriptsize[6.64, 6.73]} \\
                             &                                   & Lin.\ clip+norm           & 97.57 {\scriptsize[97.53, 97.61]} & [-2.78, 2.78]       & [-7.10, 8.07]           & 3.48 {\scriptsize[3.47, 3.50]} & 2.01 {\scriptsize[1.98, 2.05]} & 6.25 {\scriptsize[6.20, 6.30]} \\
                             &                                   & Lin.\ simplex             & 97.93 {\scriptsize[97.90, 97.97]} & [-2.57, 2.57]       & [-6.89, 7.79]           & 3.13 {\scriptsize[3.12, 3.15]} & 1.71 {\scriptsize[1.68, 1.75]} & 6.41 {\scriptsize[6.35, 6.47]} \\
                             &                                   & Vec.\ scaling             & 97.68 {\script

In [6]:
metrics_methylbert_canonicalsoft_top156 = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/cff80360fd4e412d85a39d517de0040a/artifacts/calibration/top_156_features/confidence_intervals_summary.csv"
)
print(format_deconvolver_results("MethylBERT", metrics_methylbert_canonicalsoft_top156))
del metrics_methylbert_canonicalsoft_top156

                             \multirow{20}{*}{MethylBERT}
                             & \multirow{4}{*}{XGB}              & None                      & 95.71 {\scriptsize[95.64, 95.78]} & [-3.70, 3.70]       & [-8.10, 8.71]           & 4.68 {\scriptsize[4.66, 4.70]} & 3.56 {\scriptsize[3.50, 3.63]} & 9.35 {\scriptsize[9.28, 9.41]} \\
                             &                                   & Lin.\ clip+norm           & 96.37 {\scriptsize[96.31, 96.44]} & [-3.40, 3.40]       & [-7.62, 8.32]           & 4.11 {\scriptsize[4.09, 4.13]} & 3.01 {\scriptsize[2.95, 3.07]} & 9.04 {\scriptsize[8.97, 9.11]} \\
                             &                                   & Lin.\ simplex             & 97.04 {\scriptsize[96.98, 97.10]} & [-3.07, 3.07]       & [-7.31, 7.81]           & 3.62 {\scriptsize[3.60, 3.64]} & 2.46 {\scriptsize[2.41, 2.51]} & 10.08 {\scriptsize[9.99, 10.17]} \\
                             &                                   & Vec.\ scaling             & 97.26 {\

## Pooling sensitivity analysis

In [24]:
ROOT_DIR = Path(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/823851274220463784/"
)
exp_id_per_min_reads_per_threshold = {
    15: {0.41: "4382c5e43d6b493fb200e2a2c305290f"},
    30: {
        0.2: "8a575a3aea5e406c934ffc12d77e72d3",
        0.41: "460a5d98cd3d480796e907f18b54ee1d",
        0.6: "d50b2b57053a4dc5a372dd9a447a5d46",
    },
    45: {0.41: "2c96aecb0ccb4110aa4cd096d8fd3db3"},
}
metrics_per_min_reads_per_threshold = {}
for min_reads in unique_min_reads:
    metrics_per_min_reads_per_threshold[min_reads] = {}
    for threshold in unique_thresholds:
        exp_id = exp_id_per_min_reads_per_threshold.get(min_reads, {}).get(threshold)
        if exp_id is None:
            continue
        metrics_path = (
            ROOT_DIR
            / exp_id
            / "artifacts"
            / "calibration"
            / "top_156_features"
            / "confidence_intervals_summary.csv"
        )
        if not metrics_path.exists():
            print(
                f"Metrics file not found for min_reads={min_reads}, threshold={threshold}"
            )
            continue
        metrics = pd.read_csv(metrics_path)
        metrics_per_min_reads_per_threshold[min_reads][threshold] = metrics

metrics_per_min_reads_per_threshold[30][0.41] = pd.read_csv(
    "/staging/leuven/stg_00118/gbw-d-l0154/mnt/data/syto_experiments/mlflow/198121009408531961/1f0aaa3947bb4732a945e76aed3a5864/artifacts/calibration/confidence_intervals_summary.csv"
)

unique_min_reads = sorted(exp_id_per_min_reads_per_threshold.keys())
unique_thresholds = sorted(
    {
        threshold
        for thresholds in exp_id_per_min_reads_per_threshold.values()
        for threshold in thresholds
    }
)

lines = [r"| - | $\delta=0.2$ |  $\delta=0.41$ | $\delta=0.6$ |"]
lines.append("| :---: | :---: | :---: | :---: |")
metric = "mse"
multiplier = 1e4
deconvolver = "xgb"
calib_method = "uncalibrated"  # "uncalibrated"
for min_reads in unique_min_reads:
    line = [f"| Min reads = {min_reads} |"]
    for threshold in unique_thresholds:
        if threshold not in metrics_per_min_reads_per_threshold.get(min_reads, {}):
            line.append(" - |")
            continue
        metrics = metrics_per_min_reads_per_threshold[min_reads][threshold]
        metrics_filtered = metrics.loc[
            (metrics["deconvolver"] == deconvolver)
            & (metrics["calibration_method"] == calib_method)
        ]

        metric_value = metrics_filtered[metric].values[0] * multiplier
        metric_ci_lower = metrics_filtered[f"{metric}_ci_lower"].values[0] * multiplier
        metric_ci_upper = metrics_filtered[f"{metric}_ci_upper"].values[0] * multiplier

        line.append(
            f" {metric_value:.2f} [{metric_ci_lower:.2f}, {metric_ci_upper:.2f}] |"
        )
    lines.append("".join(line))
print("\n".join(lines))

| - | $\delta=0.2$ |  $\delta=0.41$ | $\delta=0.6$ |
| :---: | :---: | :---: | :---: |
| Min reads = 15 | - | 2.10 [2.06, 2.14] | - |
| Min reads = 30 | 1.98 [1.95, 2.01] | 2.12 [2.08, 2.16] | 2.27 [2.23, 2.32] |
| Min reads = 45 | - | 2.18 [2.15, 2.22] | - |


In [20]:
for min_reads in unique_min_reads:
    for threshold in unique_thresholds:
        if threshold not in metrics_per_min_reads_per_threshold.get(min_reads, {}):
            continue
        print(format_deconvolver_results(rf"\shortstack[l]{{$\delta={threshold}$,\\$\tau={min_reads}$}}", metrics_per_min_reads_per_threshold[min_reads][threshold]))
        print(r"\midrule")

                             \multirow{20}{*}{\shortstack[l]{$\delta=0.41$,\\$\tau=15$}}
                             & \multirow{4}{*}{XGB}              & None                      & 97.47 {\scriptsize[97.43, 97.52]} & [-2.84, 2.84]       & [-7.13, 7.85]           & 3.62 {\scriptsize[3.60, 3.64]} & 2.10 {\scriptsize[2.06, 2.14]} & 6.21 {\scriptsize[6.17, 6.25]} \\
                             &                                   & Lin.\ clip+norm           & 97.84 {\scriptsize[97.80, 97.88]} & [-2.62, 2.62]       & [-6.86, 7.55]           & 3.21 {\scriptsize[3.20, 3.23]} & 1.79 {\scriptsize[1.75, 1.83]} & 5.93 {\scriptsize[5.88, 5.98]} \\
                             &                                   & Lin.\ simplex             & 98.15 {\scriptsize[98.10, 98.19]} & [-2.43, 2.43]       & [-6.68, 7.25]           & 2.90 {\scriptsize[2.89, 2.91]} & 1.54 {\scriptsize[1.50, 1.58]} & 6.23 {\scriptsize[6.17, 6.30]} \\
                             &                                   & Vec.\ s

In [42]:
metric = "mse"
multiplier = 1e4
calibration_method = "uncalibrated"  # "uncalibrated"
first_header_line = [r"\multirow{2}{*}{\textbf{Deconvolver}}"]
second_header_line = [""]
for min_reads in unique_min_reads:
    for threshold in unique_thresholds:
        if threshold not in metrics_per_min_reads_per_threshold.get(min_reads, {}):
            continue
        first_header_line.append(rf"$\mathbf{{\boldsymbol{{\delta}}={threshold}}}$")
        second_header_line.append(rf"$\mathbf{{\boldsymbol{{\tau}}={min_reads}}}$")
print("&".join(first_header_line)+r"\\")
print("&".join(second_header_line)+r"\\")
print(r"\midrule")
for deconvolver in ["nnls", "psls", "mlp", "swn", "xgb"]:
    line = [f"{deconv_map[deconvolver]}"]
    best_value = np.inf
    second_best_value = np.inf
    for min_reads in unique_min_reads:
        for threshold in unique_thresholds:
            if threshold not in metrics_per_min_reads_per_threshold.get(min_reads, {}):
                continue
            metrics = metrics_per_min_reads_per_threshold[min_reads][threshold]
            metrics = metrics.loc[
                (metrics["deconvolver"] == deconvolver)
                & (metrics["calibration_method"] == calibration_method)
            ]
            metric_value = metrics[metric].values[0]
            if metric_value < best_value:
                second_best_value = best_value
                best_value = metric_value
    
    for min_reads in unique_min_reads:
        for threshold in unique_thresholds:
            if threshold not in metrics_per_min_reads_per_threshold.get(min_reads, {}):
                continue
            metrics = metrics_per_min_reads_per_threshold[min_reads][threshold]
            metrics = metrics.loc[
                (metrics["deconvolver"] == deconvolver)
                & (metrics["calibration_method"] == calibration_method)
            ]
            metric_value = metrics[metric].values[0]
            metric_ci_lower = metrics[f"{metric}_ci_lower"].values[0]
            metric_ci_upper = metrics[f"{metric}_ci_upper"].values[0]
            line.append(
                f"{metric_value*multiplier:.2f} \\scriptsize[{metric_ci_lower*multiplier:.2f}, {metric_ci_upper*multiplier:.2f}]"
            )
            if metric_value == best_value:
                line[-1] = r"\textbf{" + line[-1] + "}"
            if metric_value == second_best_value:
                line[-1] = r"\underline{" + line[-1] + "}"

    print("&".join(line)+r"\\")

\multirow{2}{*}{\textbf{Deconvolver}}&$\mathbf{\boldsymbol{\delta}=0.41}$&$\mathbf{\boldsymbol{\delta}=0.2}$&$\mathbf{\boldsymbol{\delta}=0.41}$&$\mathbf{\boldsymbol{\delta}=0.6}$&$\mathbf{\boldsymbol{\delta}=0.41}$\\
&$\mathbf{\boldsymbol{\tau}=15}$&$\mathbf{\boldsymbol{\tau}=30}$&$\mathbf{\boldsymbol{\tau}=30}$&$\mathbf{\boldsymbol{\tau}=30}$&$\mathbf{\boldsymbol{\tau}=45}$\\
\midrule
NNLS&\underline{1.53 \scriptsize[1.50, 1.55]}&1.65 \scriptsize[1.62, 1.68]&\textbf{1.47 \scriptsize[1.45, 1.50]}&1.66 \scriptsize[1.64, 1.69]&1.61 \scriptsize[1.58, 1.64]\\
PSLS&\underline{1.60 \scriptsize[1.58, 1.63]}&1.79 \scriptsize[1.75, 1.82]&\textbf{1.55 \scriptsize[1.52, 1.57]}&1.73 \scriptsize[1.70, 1.76]&1.70 \scriptsize[1.67, 1.74]\\
MLP&\underline{2.16 \scriptsize[2.11, 2.20]}&\textbf{1.60 \scriptsize[1.58, 1.62]}&2.17 \scriptsize[2.14, 2.19]&2.22 \scriptsize[2.19, 2.25]&1.91 \scriptsize[1.89, 1.93]\\
SWN&\underline{1.24 \scriptsize[1.23, 1.26]}&\textbf{1.20 \scriptsize[1.19, 1.22]}&1.40 \scr